# Financial Risk & Anomaly Detection — Exploratory Analysis

This notebook walks through the full pipeline interactively:
1. Generate synthetic transaction data
2. Explore data characteristics
3. Engineer features
4. Run the ensemble detector
5. Evaluate and visualise results

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('Imports OK')

## 1. Generate Synthetic Transactions

In [ ]:
from src.data_generator import generate_transactions

df = generate_transactions(n_accounts=80, days=180, seed=42)

print(f'Shape: {df.shape}')
print(f'Anomaly rate: {df["true_label"].mean()*100:.1f}%')
print(f'\nAnomalies by type:')
print(df[df['true_label']==1]['anomaly_type'].value_counts())
df.head()

## 2. Basic Exploratory Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8), facecolor='#0d0f14')
fig.suptitle('Transaction Dataset Overview', color='white', fontsize=14)

# Amount distribution (log scale)
ax = axes[0, 0]
ax.set_facecolor('#13161e')
normal_amounts  = df.loc[df['true_label']==0, 'amount_gbp']
anomaly_amounts = df.loc[df['true_label']==1, 'amount_gbp']
ax.hist(normal_amounts,  bins=60, alpha=0.7, color='#3498db', label='Normal',  log=True)
ax.hist(anomaly_amounts, bins=60, alpha=0.8, color='#e74c3c', label='Anomaly', log=True)
ax.set_xlabel('Amount (GBP)', color='#7b8197')
ax.set_ylabel('Count (log)', color='#7b8197')
ax.set_title('Amount Distribution', color='white')
ax.legend()
ax.tick_params(colors='#7b8197')

# Category breakdown
ax2 = axes[0, 1]
ax2.set_facecolor('#13161e')
cat_counts = df['category'].value_counts().head(10)
ax2.barh(cat_counts.index, cat_counts.values, color='#3498db', alpha=0.8)
ax2.set_title('Top Transaction Categories', color='white')
ax2.tick_params(colors='#7b8197')

# Hourly distribution
ax3 = axes[1, 0]
ax3.set_facecolor('#13161e')
df['hour'] = pd.to_datetime(df['timestamp']).dt.hour
normal_hours  = df.loc[df['true_label']==0, 'hour'].value_counts().sort_index()
anomaly_hours = df.loc[df['true_label']==1, 'hour'].value_counts().sort_index()
ax3.bar(normal_hours.index,  normal_hours.values,  alpha=0.7, color='#3498db', label='Normal')
ax3.bar(anomaly_hours.index, anomaly_hours.values * 5, alpha=0.8, color='#e74c3c', label='Anomaly ×5')
ax3.set_xlabel('Hour of Day', color='#7b8197')
ax3.set_title('Transactions by Hour', color='white')
ax3.legend()
ax3.tick_params(colors='#7b8197')

# Currency breakdown
ax4 = axes[1, 1]
ax4.set_facecolor('#13161e')
curr = df['currency'].value_counts()
colors = ['#3498db', '#e74c3c', '#27ae60', '#f39c12', '#9b59b6', '#1abc9c', '#e67e22', '#95a5a6']
ax4.pie(curr.values[:8], labels=curr.index[:8], colors=colors,
        autopct='%1.1f%%', textprops={'color': '#e8eaf0', 'fontsize': 8})
ax4.set_title('Currency Distribution', color='white')

plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
from src.features.feature_engineering import build_features, NUMERIC_FEATURES

df_feat = build_features(df)
print(f'Features engineered: {len(NUMERIC_FEATURES)}')
print(f'Total columns: {len(df_feat.columns)}')
print(f'\nFeature groups:')
print('  Temporal:     hour_sin, hour_cos, dow_sin, dow_cos, is_weekend, is_night...')
print('  Behavioural:  amount_zscore, amount_mean_ratio, rolling_7d_ratio...')
print('  Velocity:     vel_1h, vel_6h, vel_24h, amt_vel_1h, amt_vel_24h...')
print('  Network:      acct_merchant_diversity, is_new_merchant...')
print('  Device:       is_known_device, device_switched...')
print('  Compliance:   flag_over_10k, flag_night_tx, compliance_score...')

# Show feature stats
df_feat[NUMERIC_FEATURES].describe().T[['mean','std','min','max']].round(3)

## 4. Run Ensemble Detector

In [ ]:
from src.detectors.ensemble_detector import EnsembleAnomalyDetector

detector = EnsembleAnomalyDetector(contamination=0.05, score_threshold=0.45)

print('Fitting ensemble (5 detectors)...')
results = detector.detect(df)
print(f'Done! Shape: {results.shape}')

summary = detector.summary(results)
for k, v in summary.items():
    print(f'  {k:<30} {v}')

## 5. Evaluate Performance

In [ ]:
metrics = detector.evaluate(results)

print('Performance Metrics:')
print(f'  Precision       : {metrics["precision"]:.4f}')
print(f'  Recall          : {metrics["recall"]:.4f}')
print(f'  F1 Score        : {metrics["f1_score"]:.4f}')
print(f'  ROC-AUC         : {metrics["roc_auc"]:.4f}')
print(f'  Avg Precision   : {metrics["avg_precision"]:.4f}')
print(f'  True Positives  : {metrics["true_positives"]}')
print(f'  False Positives : {metrics["false_positives"]}')

if 'per_type' in metrics:
    print('\nDetection by Anomaly Type:')
    for atype, info in metrics['per_type'].items():
        bar = '█' * int(info['recall'] * 20)
        print(f'  {atype:<25} {bar:<20} {info["recall"]*100:.0f}%')

## 6. Visualisations

In [ ]:
from src.visualization.plots import (
    plot_score_distribution,
    plot_roc_pr,
    plot_temporal_heatmap,
    plot_detector_agreement,
    plot_account_risk_scatter,
)

import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Generate and display plots inline
paths = {}
paths['score_dist']  = plot_score_distribution(results, out_dir='/tmp')
paths['roc_pr']      = plot_roc_pr(results, out_dir='/tmp')
paths['heatmap']     = plot_temporal_heatmap(results, out_dir='/tmp')
paths['agreement']   = plot_detector_agreement(results, out_dir='/tmp')
paths['acct_risk']   = plot_account_risk_scatter(results, out_dir='/tmp')

for name, path in paths.items():
    if path:
        img = mpimg.imread(path)
        plt.figure(figsize=(14, 6))
        plt.imshow(img)
        plt.axis('off')
        plt.title(name, color='white')
        plt.show()

## 7. Top Anomalies

In [ ]:
top_cols = ['transaction_id','account_id','timestamp','merchant',
            'amount_gbp','currency','risk_level','ensemble_score','alert_reason','anomaly_type']
top_cols = [c for c in top_cols if c in results.columns]

top_anomalies = (
    results[results['is_anomaly']==1][top_cols]
    .sort_values('ensemble_score', ascending=False)
    .head(15)
    .reset_index(drop=True)
)

pd.set_option('display.max_colwidth', 60)
top_anomalies